# Eviction Policies
> Control which turns are removed when conversation memory overflows.

When a `SlidingWindowMemory` exceeds its token budget it needs an eviction
policy to decide which turns to drop. Anchor ships three built-in policies:
**FIFO**, **Importance**, and **Paired**.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.memory import SlidingWindowMemory
from anchor.memory.eviction import (
    FIFOEviction,
    ImportanceEviction,
    PairedEviction,
)

## Helper: Fill a Memory
We will reuse this function to populate memory objects for each policy.

In [ ]:
def fill_memory(memory, n=8):
    """Add n user/assistant turn pairs."""
    for i in range(n):
        memory.add_turn("user", f"User message {i}: " + "x" * 30)
        memory.add_turn("assistant", f"Reply {i}: " + "y" * 30)
    return memory

print("Helper ready.")

## FIFO Eviction
The default: oldest turns are evicted first.

In [ ]:
fifo = FIFOEviction()
memory_fifo = fill_memory(SlidingWindowMemory(max_tokens=512, eviction_policy=fifo))

print(f"Turns remaining: {len(memory_fifo.turns)}")
print(f"Tokens: {memory_fifo.total_tokens} / {memory_fifo.max_tokens}")
print("\nSurviving turns (newest):")
for t in memory_fifo.turns:
    print(f"  {t.role}: {t.content[:50]}")

## Importance-Based Eviction
Provide a scoring function; turns with the **lowest** importance are evicted first.

In [ ]:
# Longer content = more important (toy heuristic)
importance_fn = lambda turn: len(turn.content) / 100
importance = ImportanceEviction(importance_fn=importance_fn)

memory_imp = fill_memory(
    SlidingWindowMemory(max_tokens=512, eviction_policy=importance)
)

print(f"Turns remaining: {len(memory_imp.turns)}")
print(f"Tokens: {memory_imp.total_tokens} / {memory_imp.max_tokens}")
print("\nSurviving turns (most important):")
for t in memory_imp.turns:
    print(f"  {t.role}: {t.content[:50]}")

## Paired Eviction
Evict user + assistant turns as a pair so you never leave an orphaned message.

In [ ]:
paired = PairedEviction()
memory_paired = fill_memory(
    SlidingWindowMemory(max_tokens=512, eviction_policy=paired)
)

print(f"Turns remaining: {len(memory_paired.turns)}")
print(f"Tokens: {memory_paired.total_tokens} / {memory_paired.max_tokens}")
print("\nSurviving paired turns:")
for t in memory_paired.turns:
    print(f"  {t.role}: {t.content[:50]}")

## Compare Side-by-Side

In [ ]:
print(f"{'Policy':<20} {'Remaining Turns':>15} {'Tokens Used':>12}")
print("-" * 50)
for name, mem in [
    ("FIFO", memory_fifo),
    ("Importance", memory_imp),
    ("Paired", memory_paired),
]:
    print(f"{name:<20} {len(mem.turns):>15} {mem.total_tokens:>12}")

## Key Takeaways
- **FIFOEviction**: simple oldest-first removal (the default)
- **ImportanceEviction**: uses a scoring function to keep high-value turns
- **PairedEviction**: preserves user/assistant turn pairs
- Pass the policy via `eviction_policy=` to `SlidingWindowMemory`

**Next:** [Decay Strategies](06_decay_strategies.ipynb)